# Análise Exploratória dos Dados

## Dicionário de dados

In [1]:
# importar os pacotes necessários
import pandas as pd
# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
dicionario = pd.read_csv("../data/raw/dicionario_dados.csv", sep=";")
dicionario.head(70)

,base,coluna,descricao
0,base_cadastral,id_cliente,Identificador único do cliente
1,base_cadastral,sexo,Sexo do cliente (M ou F)
2,base_cadastral,data_nascimento,Data de nascimento do cliente
3,base_cadastral,qtd_filhos,Quantidade de filhos declarados
4,base_cadastral,qtd_membros_familia,Quantidade total de membros da família
...,...,...,...
64,historico_parcelas,numero_parcela,Número sequencial da parcela dentro do contrato
65,historico_parcelas,data_prevista_pagamento,Data inicialmente prevista para o pagamento da...
66,historico_parcelas,data_real_pagamento,Data real em que o pagamento foi efetuado
67,historico_parcelas,valor_previsto_parcela,Valor original previsto da parcela


## Funções

In [3]:
def agg_hist_contract(hist_contratos, hist_parcelas):
    # unir historico de contratos com parcelas
    hist_total = hist_contratos.merge(
        hist_parcelas, 
        how="left", 
        on=["id_contrato", "id_cliente"]
    )

    hist_total["dias_atraso_parcela"] = (pd.to_datetime(hist_total["data_real_pagamento"]) - pd.to_datetime(hist_total["data_prevista_pagamento"])).dt.days

    hist_total = hist_total.fillna(-999)

    # CRIACAO DA VARIÁVEL ALVO DE ACORDO COM O REGULADOR (BACEN)
    hist_total["target"] = np.where(
        hist_total["dias_atraso_parcela"] > 90,
        1,
        0
    )

    features_temp = hist_total.groupby(
        [
            "id_cliente",
            "id_contrato", 
            "tipo_contrato", 
            "status_contrato", 
            "data_decisao", 
            "valor_credito", 
            "valor_bem",
            "valor_parcela",
            "percentual_entrada",
            "qtd_parcelas_planejadas",
            "tipo_pagamento",
            "finalidade_emprestimo",
            "tipo_cliente",
            "faixa_rendimento",
            "tipo_portfolio",
            "tipo_produto",
            "categoria_bem",
            "combinacao_produto",
            "setor_vendedor",
            "dia_semana_solicitacao",
            "hora_solicitacao",
            "flag_ultima_solicitacao_contrato",
            "flag_ultima_solicitacao_dia",
            "motivo_recusa",
            "acompanhantes_cliente",
            "flag_seguro_contratado"
        ]
    ).agg(
    num_max_parcelas=("numero_parcela", "max"),
    valor_total_a_pagar=("valor_parcela", "sum"),
    valor_medio_pago_parcela=("valor_pago_parcela", "mean"),
    valor_max_pago_parcela=("valor_pago_parcela", "max"),
    target=("target", "max"),
    ).reset_index()

    features_temp = features_temp.sort_values(by=["id_cliente", "data_decisao"])
    return features_temp

## Criação das Métricas Históricas

In [4]:
# importar as tabelas
cadastral = pd.read_parquet('../data/raw/base_cadastral.parquet', engine="fastparquet")
submissao = pd.read_parquet('../data/raw/base_submissao.parquet', engine="fastparquet") 
hist_contratos = pd.read_parquet('../data/raw/historico_emprestimos.parquet', engine="fastparquet")
hist_parcelas = pd.read_parquet('../data/raw/historico_parcelas.parquet', engine="fastparquet")

In [5]:
submissao.head()

,id_cliente,data_solicitacao,dia_semana_solicitacao,hora_solicitacao,tipo_contrato,valor_credito,valor_bem,valor_parcela
0,100023,2025-02-24,MONDAY,12,Cash loans,544491.0,454500.0,17563.5
1,100031,2025-02-17,MONDAY,9,Cash loans,979992.0,702000.0,27076.5
2,100056,2025-02-20,THURSDAY,10,Cash loans,1506816.0,1350000.0,49927.5
3,100069,2025-02-10,MONDAY,11,Cash loans,640458.0,517500.0,27265.5
4,100085,2025-02-19,WEDNESDAY,12,Cash loans,755190.0,675000.0,28894.5


### 1. Agregar Histórico por Contrato

In [6]:
df_hist_contract = agg_hist_contract(hist_contratos, hist_parcelas)
df_hist_contract

,id_cliente,id_contrato,tipo_contrato,status_contrato,data_decisao,valor_credito,valor_bem,valor_parcela,percentual_entrada,qtd_parcelas_planejadas,tipo_pagamento,finalidade_emprestimo,tipo_cliente,faixa_rendimento,tipo_portfolio,tipo_produto,categoria_bem,combinacao_produto,setor_vendedor,dia_semana_solicitacao,hora_solicitacao,flag_ultima_solicitacao_contrato,flag_ultima_solicitacao_dia,motivo_recusa,acompanhantes_cliente,flag_seguro_contratado,num_max_parcelas,valor_total_a_pagar,valor_medio_pago_parcela,valor_max_pago_parcela,target
0,100023,1131053,Consumer loans,Approved,2018-07-07,76680.0,78705.0,8519.130,0.101380,10.0,Cash through the bank,XAP,Repeater,low_normal,POS,XNA,Consumer Electronics,POS household without interest,Consumer electronics,SATURDAY,10,Y,1,XAP,"Spouse, partner",1.0,8.0,68153.040,10575.292500,24968.430,0
2,100023,2411919,Consumer loans,Approved,2020-02-01,93145.5,91282.5,7992.000,0.095959,16.0,Cash through the bank,XAP,Repeater,middle,POS,XNA,Consumer Electronics,POS household with interest,Consumer electronics,SATURDAY,10,Y,1,XAP,Other_B,0.0,14.0,111888.000,9046.992857,22761.900,0
1,100023,1499902,Revolving loans,Approved,2024-04-03,45000.0,45000.0,2250.000,-999.000000,0.0,XNA,XAP,Refreshed,XNA,Cards,walk-in,XNA,Card Street,XNA,WEDNESDAY,11,Y,1,XAP,-999,0.0,-999.0,2250.000,-999.000000,-999.000,0
3,100023,2454202,Cash loans,Approved,2024-06-07,239242.5,180000.0,16822.440,-999.000000,24.0,Cash through the bank,XNA,Repeater,high,Cash,x-sell,XNA,Cash X-Sell: high,XNA,FRIDAY,9,Y,1,XAP,Unaccompanied,1.0,4.0,67289.760,70571.711250,231819.525,0
4,100056,1636219,Consumer loans,Approved,2017-05-25,201528.0,223920.0,13502.385,0.108909,24.0,Cash through the bank,XAP,New,high,POS,XNA,Computers,POS household with interest,Consumer electronics,THURSDAY,8,Y,1,XAP,-999,0.0,101.0,189033.390,22716.144643,168149.790,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186885,456245,1666363,Cash loans,Approved,2024-07-12,116077.5,90000.0,6481.755,-999.000000,24.0,Cash through the bank,XNA,Repeater,low_normal,Cash,x-sell,XNA,Cash X-Sell: low,XNA,FRIDAY,10,Y,1,XAP,"Spouse, partner",1.0,7.0,45372.285,6481.755000,6481.755,0
186887,456248,1395578,Consumer loans,Approved,2017-10-05,41940.0,39960.0,8417.340,0.000000,6.0,Cash through the bank,XAP,New,high,POS,XNA,Mobile,POS mobile with interest,Connectivity,THURSDAY,14,Y,1,XAP,Family,1.0,6.0,84173.400,5047.740000,8417.340,0
186888,456248,1826280,Cash loans,Approved,2022-01-10,271300.5,225000.0,28633.320,-999.000000,12.0,Cash through the bank,XNA,Refreshed,middle,Cash,x-sell,XNA,Cash X-Sell: middle,XNA,MONDAY,13,Y,1,XAP,Family,1.0,12.0,343599.840,28608.903750,28633.320,0
186886,456248,1136073,Cash loans,Approved,2023-01-27,1333179.0,1260000.0,67725.495,-999.000000,24.0,Cash through the bank,XNA,Repeater,low_action,Cash,x-sell,XNA,Cash X-Sell: low,XNA,FRIDAY,16,Y,1,XAP,-999,1.0,24.0,1625411.880,67709.353125,67725.495,0


### 2. Criar Janela Temporal Agregando as Métricas por Janela de "data_decisao"

In [7]:
# ==============================
# CONFIG
# ==============================
COLUNAS_ULTIMO_VALOR = [
    "tipo_contrato",
    "status_contrato",
    "tipo_pagamento",
    "finalidade_emprestimo",
    "tipo_cliente",
    "faixa_rendimento",
    "tipo_portfolio",
    "tipo_produto",
    "categoria_bem",
    "combinacao_produto",
    "setor_vendedor",
    "motivo_recusa",
    "acompanhantes_cliente",
    "flag_seguro_contratado",
    "flag_ultima_solicitacao_contrato",
    "flag_ultima_solicitacao_dia",
    "dia_semana_solicitacao",
    "hora_solicitacao",
]

AGGS = {
    "valor_credito": ["mean", "max", "min"],
    "valor_bem": ["mean", "max", "min"],
    "valor_parcela": ["mean", "max", "min"],
    "percentual_entrada": ["mean"],
    "qtd_parcelas_planejadas": ["max"],
    "num_max_parcelas": ["mean", "max"],
    "valor_total_a_pagar": ["mean", "sum"],
    "valor_medio_pago_parcela": ["mean"],
    "valor_max_pago_parcela": ["max"],
}

COLUNAS_REMOVER = [
    "status_contrato",
    "percentual_entrada",
    "qtd_parcelas_planejadas",
    "tipo_pagamento",
    "finalidade_emprestimo",
    "tipo_cliente",
    "faixa_rendimento",
    "tipo_portfolio",
    "tipo_produto",
    "categoria_bem",
    "combinacao_produto",
    "setor_vendedor",
    "flag_ultima_solicitacao_contrato",
    "flag_ultima_solicitacao_dia",
    "motivo_recusa",
    "acompanhantes_cliente",
    "flag_seguro_contratado",
    "num_max_parcelas",
    "valor_total_a_pagar",
    "valor_medio_pago_parcela",
    "valor_max_pago_parcela",
    "hist_ultimo_dia_semana_solicitacao",
    "hist_ultimo_hora_solicitacao"
]

# ==============================
# FUNÇÃO PRINCIPAL
# ==============================

def create_rolling_features(df: pd.DataFrame, colunas_remover: list) -> pd.DataFrame:
    
    df = df.sort_values(["id_cliente", "data_decisao"]).copy()
    group = df.groupby("id_cliente")

    # ==============================
    # QUANTIDADE DE CONTRATOS
    # ==============================
    df["hist_qtd_contratos"] = group.cumcount().replace(0, np.nan)

    # ==============================
    # FEATURES NUMÉRICAS (EXPANDING)
    # ==============================
    for col, funcs in AGGS.items():
        tmp = (
            group[col]
            .expanding()
            .agg(funcs)
            .shift(1)
            .reset_index(level=0, drop=True)
        )

        # Caso venha como Series (apenas 1 agg)
        if isinstance(tmp, pd.Series):
            df[f"hist_{col}_{funcs[0]}"] = tmp
        else:
            for f in funcs:
                df[f"hist_{col}_{f}"] = tmp[f]

    # ==============================
    # ÚLTIMO VALOR (SHIFT)
    # ==============================
    for col in COLUNAS_ULTIMO_VALOR:
        df[f"hist_ultimo_{col}"] = group[col].shift(1)

    # ==============================
    # GARANTIA: PRIMEIRA LINHA = NaN
    # ==============================
    first_idx = group.head(1).index
    cols_hist = [c for c in df.columns if c.startswith("hist_")]

    df.loc[first_idx, cols_hist] = np.nan
    df = df.drop(columns=colunas_remover, errors="ignore")

    return df

# ==============================
# EXECUÇÃO
# ==============================

df_window = create_rolling_features(df_hist_contract, COLUNAS_REMOVER)

In [8]:
df_window

,id_cliente,id_contrato,tipo_contrato,data_decisao,valor_credito,valor_bem,valor_parcela,dia_semana_solicitacao,hora_solicitacao,target,hist_qtd_contratos,hist_valor_credito_mean,hist_valor_credito_max,hist_valor_credito_min,hist_valor_bem_mean,hist_valor_bem_max,hist_valor_bem_min,hist_valor_parcela_mean,hist_valor_parcela_max,hist_valor_parcela_min,hist_percentual_entrada_mean,hist_qtd_parcelas_planejadas_max,hist_num_max_parcelas_mean,hist_num_max_parcelas_max,hist_valor_total_a_pagar_mean,hist_valor_total_a_pagar_sum,hist_valor_medio_pago_parcela_mean,hist_valor_max_pago_parcela_max,hist_ultimo_tipo_contrato,hist_ultimo_status_contrato,hist_ultimo_tipo_pagamento,hist_ultimo_finalidade_emprestimo,hist_ultimo_tipo_cliente,hist_ultimo_faixa_rendimento,hist_ultimo_tipo_portfolio,hist_ultimo_tipo_produto,hist_ultimo_categoria_bem,hist_ultimo_combinacao_produto,hist_ultimo_setor_vendedor,hist_ultimo_motivo_recusa,hist_ultimo_acompanhantes_cliente,hist_ultimo_flag_seguro_contratado,hist_ultimo_flag_ultima_solicitacao_contrato,hist_ultimo_flag_ultima_solicitacao_dia
0,100023,1131053,Consumer loans,2018-07-07,76680.0,78705.0,8519.130,SATURDAY,10,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100023,2411919,Consumer loans,2020-02-01,93145.5,91282.5,7992.000,SATURDAY,10,0,1.0,76680.00,76680.0,76680.0,78705.00,78705.0,78705.0,8519.130,8519.130,8519.130,0.101380,10.0,8.000000,8.0,68153.04,68153.04,10575.292500,24968.430,Consumer loans,Approved,Cash through the bank,XAP,Repeater,low_normal,POS,XNA,Consumer Electronics,POS household without interest,Consumer electronics,XAP,"Spouse, partner",1.0,Y,1.0
1,100023,1499902,Revolving loans,2024-04-03,45000.0,45000.0,2250.000,WEDNESDAY,11,0,2.0,84912.75,93145.5,76680.0,84993.75,91282.5,78705.0,8255.565,8519.130,7992.000,0.098669,16.0,11.000000,14.0,90020.52,180041.04,9811.142679,24968.430,Consumer loans,Approved,Cash through the bank,XAP,Repeater,middle,POS,XNA,Consumer Electronics,POS household with interest,Consumer electronics,XAP,Other_B,0.0,Y,1.0
3,100023,2454202,Cash loans,2024-06-07,239242.5,180000.0,16822.440,FRIDAY,9,0,3.0,71608.50,93145.5,45000.0,71662.50,91282.5,45000.0,6253.710,8519.130,2250.000,-332.934220,16.0,-325.666667,14.0,60763.68,182291.04,6207.761786,24968.430,Revolving loans,Approved,XNA,XAP,Refreshed,XNA,Cards,walk-in,XNA,Card Street,XNA,XAP,-999,0.0,Y,1.0
4,100056,1636219,Consumer loans,2017-05-25,201528.0,223920.0,13502.385,THURSDAY,8,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186885,456245,1666363,Cash loans,2024-07-12,116077.5,90000.0,6481.755,FRIDAY,10,0,1.0,47970.00,47970.0,47970.0,45000.00,45000.0,45000.0,5693.085,5693.085,5693.085,-999.000000,12.0,12.000000,12.0,68317.02,68317.02,5692.650000,5693.085,Cash loans,Approved,Cash through the bank,Urgent needs,New,high,Cash,walk-in,XNA,Cash Street: high,Connectivity,XAP,-999,1.0,Y,1.0
186887,456248,1395578,Consumer loans,2017-10-05,41940.0,39960.0,8417.340,THURSDAY,14,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
186888,456248,1826280,Cash loans,2022-01-10,271300.5,225000.0,28633.320,MONDAY,13,0,1.0,41940.00,41940.0,41940.0,39960.00,39960.0,39960.0,8417.340,8417.340,8417.340,0.000000,6.0,6.000000,6.0,84173.40,84173.40,5047.740000,8417.340,Consumer loans,Approved,Cash through the bank,XAP,New,high,POS,XNA,Mobile,POS mobile with interest,Connectivity,XAP,Family,1.0,Y,1.0
186886,456248,1136073,Cash loans,2023-01-27,1333179.0,1260000.0,67725.495,FRIDAY,16,0,2.0,156620.25,271300.5,41940.0,132480.00,225000.0,39960.0,18525.330,28633.320,8417.340,-499.500000,12.0,9.000000,12.0

### 3. Unir dados de janela com os dados cadastrais

In [10]:
df_window_final = df_window.merge(
    cadastral, 
    how="left", 
    on="id_cliente"
)

df_window_final = df_window_final.drop(["sexo", "data_nascimento"], axis=1)

In [11]:
df_window_final

,id_cliente,id_contrato,tipo_contrato,data_decisao,valor_credito,valor_bem,valor_parcela,dia_semana_solicitacao,hora_solicitacao,target,hist_qtd_contratos,hist_valor_credito_mean,hist_valor_credito_max,hist_valor_credito_min,hist_valor_bem_mean,hist_valor_bem_max,hist_valor_bem_min,hist_valor_parcela_mean,hist_valor_parcela_max,hist_valor_parcela_min,hist_percentual_entrada_mean,hist_qtd_parcelas_planejadas_max,hist_num_max_parcelas_mean,hist_num_max_parcelas_max,hist_valor_total_a_pagar_mean,hist_valor_total_a_pagar_sum,hist_valor_medio_pago_parcela_mean,hist_valor_max_pago_parcela_max,hist_ultimo_tipo_contrato,hist_ultimo_status_contrato,hist_ultimo_tipo_pagamento,hist_ultimo_finalidade_emprestimo,hist_ultimo_tipo_cliente,hist_ultimo_faixa_rendimento,hist_ultimo_tipo_portfolio,hist_ultimo_tipo_produto,hist_ultimo_categoria_bem,hist_ultimo_combinacao_produto,hist_ultimo_setor_vendedor,hist_ultimo_motivo_recusa,hist_ultimo_acompanhantes_cliente,hist_ultimo_flag_seguro_contratado,hist_ultimo_flag_ultima_solicitacao_contrato,hist_ultimo_flag_ultima_solicitacao_dia,qtd_filhos,qtd_membros_familia,renda_anual,tipo_renda,ocupacao,tipo_organizacao,nivel_educacao,estado_civil,tipo_moradia,possui_carro,possui_imovel,nota_regiao_cliente,nota_regiao_cliente_cidade
0,100023,1131053,Consumer loans,2018-07-07,76680.0,78705.0,8519.130,SATURDAY,10,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2.0,90000.0,State servant,Core staff,Kindergarten,Higher education,Single / not married,House / apartment,N,Y,2,2
1,100023,2411919,Consumer loans,2020-02-01,93145.5,91282.5,7992.000,SATURDAY,10,0,1.0,76680.00,76680.0,76680.0,78705.00,78705.0,78705.0,8519.130,8519.130,8519.130,0.101380,10.0,8.000000,8.0,68153.04,68153.04,10575.292500,24968.430,Consumer loans,Approved,Cash through the bank,XAP,Repeater,low_normal,POS,XNA,Consumer Electronics,POS household without interest,Consumer electronics,XAP,"Spouse, partner",1.0,Y,1.0,1,2.0,90000.0,State servant,Core staff,Kindergarten,Higher education,Single / not married,House / apartment,N,Y,2,2
2,100023,1499902,Revolving loans,2024-04-03,45000.0,45000.0,2250.000,WEDNESDAY,11,0,2.0,84912.75,93145.5,76680.0,84993.75,91282.5,78705.0,8255.565,8519.130,7992.000,0.098669,16.0,11.000000,14.0,90020.52,180041.04,9811.142679,24968.430,Consumer loans,Approved,Cash through the bank,XAP,Repeater,middle,POS,XNA,Consumer Electronics,POS household with interest,Consumer electronics,XAP,Other_B,0.0,Y,1.0,1,2.0,90000.0,State servant,Core staff,Kindergarten,Higher education,Single / not married,House / apartment,N,Y,2,2
3,100023,2454202,Cash loans,2024-06-07,239242.5,180000.0,16822.440,FRIDAY,9,0,3.0,71608.50,93145.5,45000.0,71662.50,91282.5,45000.0,6253.710,8519.130,2250.000,-332.934220,16.0,-325.666667,14.0,60763.68,182291.04,6207.761786,24968.430,Revolving loans,Approved,XNA,XAP,Refreshed,XNA,Cards,walk-in,XNA,Card Street,XNA,XAP,-999,0.0,Y,1.0,1,2.0,90000.0,State servant,Core staff,Kindergarten,Higher education,Single / not married,House / apartment,N,Y,2,2
4,100056,1636219,Consumer loans,2017-05-25,201528.0,223920.0,13502.385,THURSDAY,8,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2.0,360000.0,Working,Laborers,Transport: type 2,Secondary / secondary special,Married,House / apartment,Y,Y,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186885,456245,1666363,Cash loans,2024-07-12,116077.5,90000.0,6481.755,FRIDAY,10,0,1.0,47970.00,47970.0,47970.0,45000.00,45000.0,45000.0,5693.085,5693.085,5693.085,-999.000000,12.0,12.000000,12.0,68317.02,68317.02,5692.650000,5693.085,Cash loans,Approved,Cash through the bank,Urgent needs,New,high,Cash,walk-in,XNA,Cash Street: h

### 4. Criar tabelas de treino e teste

# TREINAMENTO DO MODELO

In [ ]:
# df_splitted = df_window.sort_values("data_decisao")

# # corte temporal (exemplo)
# cutoff_date = "2024-01-01"

# train = df_window[df_window["data_decisao"] < cutoff_date].copy()
# test  = df_window[df_window["data_decisao"] >= cutoff_date].copy()

# print(train.shape, test.shape)

(115041, 44) (71849, 44)


In [ ]:
# target = "target"

# # remover colunas que não entram no modelo
# drop_cols = ["id_cliente", "id_contrato", "data_decisao"]

# X_train = train.drop(columns=drop_cols + [target])
# y_train = train[target]

# X_test = test.drop(columns=drop_cols + [target])
# y_test = test[target]

In [ ]:
# cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
# num_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

# print("Categoricas:", cat_cols)

Categoricas: ['tipo_contrato', 'dia_semana_solicitacao', 'hist_ultimo_tipo_contrato', 'hist_ultimo_status_contrato', 'hist_ultimo_tipo_pagamento', 'hist_ultimo_finalidade_emprestimo', 'hist_ultimo_tipo_cliente', 'hist_ultimo_faixa_rendimento', 'hist_ultimo_tipo_portfolio', 'hist_ultimo_tipo_produto', 'hist_ultimo_categoria_bem', 'hist_ultimo_combinacao_produto', 'hist_ultimo_setor_vendedor', 'hist_ultimo_motivo_recusa', 'hist_ultimo_acompanhantes_cliente', 'hist_ultimo_flag_ultima_solicitacao_contrato']


In [ ]:
# import lightgbm as lgb

# # converter categóricas
# for col in cat_cols:
#     X_train[col] = X_train[col].astype("category")
#     X_test[col] = X_test[col].astype("category")

# model = lgb.LGBMClassifier(
#     n_estimators=500,
#     learning_rate=0.05,
#     random_state=42
# )

# model.fit(
#     X_train, y_train,
#     eval_set=[(X_test, y_test)],
#     eval_metric="auc",
#     categorical_feature=cat_cols,
# )

[LightGBM] [Info] Number of positive: 833, number of negative: 114208
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035145 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5011
[LightGBM] [Info] Number of data points in the train set: 115041, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.007241 -> initscore=-4.920743
[LightGBM] [Info] Start training from score -4.920743


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [ ]:
# from sklearn.metrics import roc_auc_score

# y_pred = model.predict_proba(X_test)[:, 1]

# auc = roc_auc_score(y_test, y_pred)
# print("AUC:", auc)

AUC: 0.8098757527674935


In [ ]:
y_pred

array([1.65805630e-04, 8.81499150e-04, 2.48239796e-05, ...,
       7.84083430e-04, 5.56406521e-05, 5.76355676e-03], shape=(71849,))